<a href="https://colab.research.google.com/github/UgoTheOne/Twitter-sentiment-analysis/blob/main/Twitter_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json
import os

# Paste your token content inside the quotes below
kaggle_credentials = {"username": "it is me", "key": "not_your_business"}

# This automatically creates the kaggle.json file in the correct folder
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

os.chmod('/root/.kaggle/kaggle.json', 600)
print("Kaggle credentials configured successfully!")

Kaggle credentials configured successfully!


In [5]:
!kaggle datasets download -d saurabhshahane/twitter-sentiment-dataset

Dataset URL: https://www.kaggle.com/datasets/saurabhshahane/twitter-sentiment-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 7.60M/7.60M [00:01<00:00, 4.75MB/s]



In [6]:
!unzip -q twitter-sentiment-dataset.zip -d twitter_sentiment

In [8]:
!ls twitter_sentiment

Twitter_Data.csv


In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,classification_report

In [11]:
df = pd.read_csv('twitter_sentiment/Twitter_Data.csv')
df.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


In [16]:
# This is me cleaning my dataset by dropping the NaN rows
print(df.isnull().sum())
df = df.dropna(subset=['clean_text', 'category'])

clean_text    0
category      0
dtype: int64


In [17]:
# Separating and Splitting the datasets
X = df['clean_text']
y = df['category']
# Splitting the data into training and testing sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state=42)

print(f"Training data size: {X_train.shape[0]} tweets")
print(f"Testing data size: {X_test.shape[0]} tweets")

Training data size: 130375 tweets
Testing data size: 32594 tweets


In [18]:
vectorizer = TfidfVectorizer(max_features=100000, ngram_range = (1,2))
# Learn the vocabulary from training data and transform it into numbers
X_train_vec = vectorizer.fit_transform(X_train)
#Transform test data using only the learned vocabulary
X_test_vec = vectorizer.transform(X_test)

print(f"X_train_vec shape: {X_train_vec.shape}")
print(f"X_test_vec shape:{X_test_vec.shape}")

X_train_vec shape: (130375, 100000)
X_test_vec shape:(32594, 100000)


In [20]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec,y_train)
y_pred = model.predict(X_test_vec)
print("\nAccuracy Score:", accuracy_score(y_test,y_pred))
print("\nClassification Report:\n", classification_report(y_test,y_pred))


Accuracy Score: 0.9074983125728662

Classification Report:
               precision    recall  f1-score   support

        -1.0       0.91      0.78      0.84      7152
         0.0       0.89      0.97      0.92     11067
         1.0       0.92      0.93      0.92     14375

    accuracy                           0.91     32594
   macro avg       0.91      0.89      0.90     32594
weighted avg       0.91      0.91      0.91     32594



In [26]:
my_tweets = [
    "I absolutely love this product, it works wonderfully",
    "I am very happy about my little progress in the right direction",
    "The situation of this country is bad",
    "Tunde is a good boy",
    "okay",
    "Football"
]
my_tweets_vec = vectorizer.transform(my_tweets)
predictions = model.predict(my_tweets_vec)
sentiment_map = {-1.0:"Negative", 0.0:"Neutral", 1.0:"Positive"}
for tweet, pred in zip(my_tweets, predictions):
  print(f"Tweet: '{tweet} --> Sentiment:{sentiment_map[pred]}")

Tweet: 'I absolutely love this product, it works wonderfully --> Sentiment:Positive
Tweet: 'I am very happy about my little progress in the right direction --> Sentiment:Positive
Tweet: 'The situation of this country is bad --> Sentiment:Negative
Tweet: 'Tunde is a good boy --> Sentiment:Positive
Tweet: 'okay --> Sentiment:Positive
Tweet: 'Football --> Sentiment:Neutral


In [27]:
import joblib
joblib.dump(model,"twitter_sentiment_model.joblib")
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')

['tfidf_vectorizer.joblib']

In [28]:
from google.colab import files
files.download('twitter_sentiment_model.joblib')
files.download("tfidf_vectorizer.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>